# 01 · Qué es un LLM y qué es un agente

**Python que piensa: agentes de analítica con LangChain y LangGraph**
Sociedad Ecuatoriana de Estadística · Semana de Python

---

Tres ideas, en este orden:

1. **Un LLM es un servicio web.** Se le manda texto con un POST y devuelve texto.
2. **Un LLM solo predice texto.** No sabe qué día es, no conoce tu base y no
   ejecuta nada.
3. **Un agente es un LLM con herramientas dentro de un bucle.** El modelo decide
   qué herramienta usar, Python la ejecuta, el modelo mira el resultado y vuelve
   a decidir.

Todo sin frameworks: solo `requests` y el SDK de `openai`. LangChain entra en el
notebook 02, cuando ya se vea qué trabajo nos ahorra.

> **Sobre los datos.** Reclamos médicos **sintéticos** del CMS de Estados
> Unidos. Sirven para construir herramientas, no para inferir sobre la población
> real de Medicare.

## Legibilidad para proyección

Agranda la tipografía de las salidas para que se lea desde el fondo de la sala.

In [ ]:
from IPython.display import HTML, display

display(HTML("""
<style>
  .jp-RenderedText pre, .jp-OutputArea-output pre, div.output_area pre {
      font-size: 15px !important; line-height: 1.5 !important; }
  .jp-RenderedHTMLCommon table { font-size: 15px !important; }
  .jp-InputArea-editor, .CodeMirror { font-size: 15px !important; }
</style>
"""))
print("Tipografía ajustada para proyección.")

## Dónde estamos

Rutas relativas a la raíz del repositorio, para que el notebook corra igual en
otra máquina.

In [ ]:
from pathlib import Path

RAIZ = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "datos").is_dir()), None)
if RAIZ is None:
    raise SystemExit("No encuentro la carpeta datos/. Abre Jupyter dentro del repositorio.")

BASE = RAIZ / "datos" / "reclamos.db"
print(f"Raíz del proyecto : {RAIZ}")
print(f"Base de datos     : {BASE}  {'(existe)' if BASE.exists() else '(falta: corre 00_setup.ipynb)'}")

## Las credenciales

Una **API key** es la contraseña de un programa: identifica quién llama y a
quién se le factura. Un **endpoint** es la dirección web del modelo. Las dos
viven en el archivo `.env`, que **nunca** se sube al repositorio.

Esta celda las carga y muestra cuáles están definidas, sin imprimir su valor.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv(RAIZ / ".env")

for variable in ["AZURE_OPENAI_BASE_URL", "AZURE_OPENAI_API_KEY", "AZURE_OPENAI_MODELO"]:
    print(f"{variable:<24} {'definida' if os.getenv(variable) else 'FALTA'}")

---

# Parte A · Un LLM es un POST

> **Pausa: consola de Azure AI Foundry.** Aquí se sale al navegador:
>
> 1. Abrir el proyecto en [ai.azure.com](https://ai.azure.com).
> 2. Catálogo de modelos → elegir el modelo → **Deploy**. El nombre que se le
>    pone al deployment es el que va en `AZURE_OPENAI_MODELO`.
> 3. En el deployment: copiar el **endpoint** (termina en `/openai/v1`) y la
>    **key** al `.env`.
>
> El recurso ya está desplegado antes de la charla: se muestra el proceso sobre
> uno nuevo solo hasta el botón de crear.

Con esas tres cosas, hablar con el modelo es una petición HTTP como cualquier
otra.

## La petición, cruda

Un POST con tres partes: **a dónde** (el endpoint), **quién soy** (la key en el
encabezado) y **qué quiero** (el modelo y la lista de mensajes). Nada más.

In [ ]:
import json
import requests

ENDPOINT = os.environ["AZURE_OPENAI_BASE_URL"].rstrip("/")
API_KEY  = os.environ["AZURE_OPENAI_API_KEY"]
MODELO   = os.environ["AZURE_OPENAI_MODELO"]

respuesta = requests.post(
    f"{ENDPOINT}/chat/completions",
    headers={"api-key": API_KEY},
    json={
        "model": MODELO,
        "messages": [
            {"role": "user", "content": "Explica en una frase qué es un reclamo médico."}
        ],
    },
)

datos = respuesta.json()
print(json.dumps(datos, indent=2, ensure_ascii=False))

## Qué trae la respuesta

Del JSON importan tres cosas: el **texto**, el **modelo** que respondió y los
**tokens**. Un **token** es un pedazo de palabra (unos cuatro caracteres en
promedio); es la unidad con que el proveedor cobra, por separado los de entrada
y los de salida.

In [ ]:
print("Texto            :", datos["choices"][0]["message"]["content"])
print("Modelo           :", datos["model"])
print("Tokens de entrada:", datos["usage"]["prompt_tokens"])
print("Tokens de salida :", datos["usage"]["completion_tokens"])

## Lo mismo, con el SDK

El **SDK** es una biblioteca del proveedor que arma ese POST por nosotros. Es
exactamente la misma petición, en cuatro líneas. Como el endpoint de Foundry
termina en `/openai/v1`, sirve el cliente estándar de `openai`.

In [ ]:
from openai import OpenAI

cliente = OpenAI(base_url=ENDPOINT, api_key=API_KEY)

respuesta = cliente.chat.completions.create(
    model=MODELO,
    messages=[{"role": "user", "content": "Explica en una frase qué es un reclamo médico."}],
)
print(respuesta.choices[0].message.content)

---

# Parte B · Lo que un LLM no puede hacer

Un **LLM** (modelo de lenguaje grande) predice el siguiente token a partir de lo
que vio al entrenarse. Eso tiene consecuencias directas.

## No sabe qué día es

Su conocimiento se congeló cuando terminó el entrenamiento, y no tiene reloj.

In [ ]:
respuesta = cliente.chat.completions.create(
    model=MODELO,
    messages=[{"role": "user", "content": "¿Qué fecha es hoy? Responde solo con la fecha."}],
)
print("El modelo dice:", respuesta.choices[0].message.content)

from datetime import date
print("Hoy es        :", date.today().isoformat())

## No conoce tus datos

Le preguntamos la cifra que el agente va a tener que responder. El modelo nunca
vio nuestra base: **o se niega, o inventa**. Al lado, la cifra real, consultada
directamente en la sábana.

In [ ]:
import sqlite3
import pandas as pd

pregunta = ("En mi base de reclamos médicos, ¿cuánto se pagó en hospitalización "
            "en 2009? Responde solo con la cifra.")

respuesta = cliente.chat.completions.create(
    model=MODELO,
    messages=[{"role": "user", "content": pregunta}],
)

con = sqlite3.connect(BASE)
real = con.execute(
    "SELECT sum(monto_pagado) FROM sabana WHERE ambito = 'Hospitalario' AND anio = 2009"
).fetchone()[0]
con.close()

pd.DataFrame({
    "fuente": ["El LLM, sin acceso a la base", "La base (tabla sabana)"],
    "respuesta": [respuesta.choices[0].message.content, f"{real:,.2f}"],
})

## La temperatura

La **temperatura** controla cuánto se arriesga el modelo al elegir el siguiente
token. En 0 elige casi siempre el más probable; en 1 muestrea más. Para
analítica se usa baja: queremos la misma consulta cada vez, no creatividad.

El mismo pedido, tres veces con cada temperatura.

In [ ]:
pedido = "Escribe un titular de máximo diez palabras sobre el costo de la salud."

filas = []
for temperatura in [0, 1]:
    for intento in [1, 2, 3]:
        respuesta = cliente.chat.completions.create(
            model=MODELO,
            temperature=temperatura,
            messages=[{"role": "user", "content": pedido}],
        )
        filas.append({"temperatura": temperatura, "intento": intento,
                      "titular": respuesta.choices[0].message.content})

pd.set_option("display.max_colwidth", None)
pd.DataFrame(filas)

---

# Parte C · Un agente hecho a mano

Un **agente** es un LLM, más **herramientas**, más un **bucle** que decide qué
hacer después. Lo armamos sin framework para ver que no hay magia.

Una **herramienta** (*tool*) es una función de Python que el modelo puede
**pedir** que se ejecute. El modelo no la ejecuta: escribe un pedido, y nuestro
código decide si lo cumple.

## La herramienta

Una función que recibe un SQL y devuelve las filas. Abre la base en **modo solo
lectura** (`mode=ro`): el modelo puede pedir lo que quiera, pero no puede borrar
ni modificar nada.

In [ ]:
def consultar_sql(sql):
    con = sqlite3.connect(BASE.as_uri() + "?mode=ro", uri=True)
    filas = con.execute(sql).fetchall()
    con.close()
    return filas

consultar_sql("SELECT count(*) FROM sabana")

## La descripción que lee el modelo

El modelo no ve el código de la función: ve **esta descripción**, en formato
JSON Schema. El nombre, para qué sirve y qué parámetros recibe. Si la
descripción es mala, el modelo usa mal la herramienta.

In [ ]:
herramientas = [{
    "type": "function",
    "function": {
        "name": "consultar_sql",
        "description": "Ejecuta una consulta SQL de solo lectura sobre la base "
                       "SQLite de reclamos médicos y devuelve las filas.",
        "parameters": {
            "type": "object",
            "properties": {
                "sql": {"type": "string", "description": "La consulta SQL (SQLite) a ejecutar."}
            },
            "required": ["sql"],
        },
    },
}]
print(json.dumps(herramientas, indent=2, ensure_ascii=False))

## El contexto mínimo

El modelo necesita saber qué tabla existe. Por ahora le damos **solo los nombres
de las columnas**, sin significados ni valores posibles. Ese diccionario
completo entra en el notebook 02, y ahí se va a ver la diferencia.

In [ ]:
con = sqlite3.connect(BASE)
columnas = [fila[1] for fila in con.execute("PRAGMA table_info(sabana)")]
con.close()

sistema = ("Eres un analista de datos. Responde usando la herramienta consultar_sql. "
           f"La única tabla es 'sabana', con estas columnas: {', '.join(columnas)}.")
print(sistema)

## Llamada 1 · El modelo pide la herramienta

Le mandamos la pregunta, el contexto y la lista de herramientas. **No responde
con texto**: responde con un pedido, el nombre de la herramienta y el SQL que
quiere ejecutar.

In [ ]:
mensajes = [
    {"role": "system", "content": sistema},
    {"role": "user", "content": "¿Cuánto se pagó en hospitalización en 2009?"},
]

respuesta = cliente.chat.completions.create(
    model=MODELO, temperature=0, messages=mensajes, tools=herramientas,
)
pedido_del_modelo = respuesta.choices[0].message

print("Texto      :", pedido_del_modelo.content)
print("Herramienta:", pedido_del_modelo.tool_calls[0].function.name)
print("SQL        :", json.loads(pedido_del_modelo.tool_calls[0].function.arguments)["sql"])

## Python ejecuta

Quien ejecuta es **nuestro código**, no el modelo. Aquí podríamos validar el SQL,
pedir aprobación o negarnos.

In [ ]:
llamada = pedido_del_modelo.tool_calls[0]
sql = json.loads(llamada.function.arguments)["sql"]

resultado = consultar_sql(sql)
resultado

## Llamada 2 · El modelo redacta

Le devolvemos al modelo su propio pedido y el resultado, como un mensaje de rol
`tool`. Con eso ya tiene lo que necesita y responde en lenguaje natural.

In [ ]:
mensajes.append(pedido_del_modelo)
mensajes.append({"role": "tool", "tool_call_id": llamada.id, "content": json.dumps(resultado)})

respuesta = cliente.chat.completions.create(
    model=MODELO, temperature=0, messages=mensajes, tools=herramientas,
)
print("Agente :", respuesta.choices[0].message.content)
print(f"Real   : {real:,.2f}")

## Eso, en un bucle, es un agente

Las dos llamadas de arriba se repiten hasta que el modelo deja de pedir
herramientas. **Ese `while` es el agente entero.** El modelo decide; Python
ejecuta; el modelo mira el resultado y vuelve a decidir.

Una pregunta nueva, sin tocar el código.

In [ ]:
mensajes = [
    {"role": "system", "content": sistema},
    {"role": "user", "content": "¿Qué estado tuvo el mayor costo ambulatorio en 2009?"},
]

while True:
    respuesta = cliente.chat.completions.create(
        model=MODELO, temperature=0, messages=mensajes, tools=herramientas,
    )
    mensaje = respuesta.choices[0].message
    mensajes.append(mensaje)

    if not mensaje.tool_calls:          # ya no pide nada: terminó
        break

    for llamada in mensaje.tool_calls:  # pide algo: Python lo ejecuta
        sql = json.loads(llamada.function.arguments)["sql"]
        print("El modelo pide:", sql)
        resultado = consultar_sql(sql)
        print("Resultado     :", resultado[:5], "\n")
        mensajes.append({"role": "tool", "tool_call_id": llamada.id,
                         "content": json.dumps(resultado)})

print("Agente:", mensaje.content)

---

## Cierre

> **El LLM no es el agente. El LLM es el que decide dentro del agente.**

- Un LLM es un POST: endpoint, key, mensajes.
- Solo predice texto: no sabe la fecha ni conoce tus datos.
- Un agente es ese LLM con herramientas, dentro de un bucle que nuestro código
  controla.

Lo que falta, y que resuelven los notebooks siguientes: el modelo escribió SQL
**adivinando** qué significa cada columna; si mañana cambiamos de proveedor hay
que reescribir las llamadas; y el bucle no recuerda nada ni valida si el
resultado responde la pregunta.

### Decisión de arquitectura · ¿de dónde sale el modelo?

| Camino | Cuándo conviene | Qué se gana | Qué se paga |
|---|---|---|---|
| **Azure AI Foundry** (o la nube corporativa) | Banca, seguros, sector público | El dato no sale de tu nube, facturación corporativa, catálogo de varios proveedores | Más pasos para empezar; disponibilidad de modelos por región |
| **API directa del proveedor** | Prototipos, equipos pequeños | Lo más simple; los modelos nuevos llegan primero | El dato viaja al proveedor; otra factura que gobernar |
| **Suscripción de producto** (el chat web) | Uso personal | Nada que construir | No sirve para construir: no hay API ni control |

Los criterios para elegir: **gobierno del dato**, **costo por millón de
tokens**, **latencia**, **caché de prompts** y **qué tan fácil es cambiar de
modelo**. Los tres últimos se miden en el notebook 02.

### Tres preguntas de entrevista

1. En el bucle del agente, ¿quién ejecuta la herramienta, el modelo o tu código?
   ¿Qué controles pondrías en ese punto antes de correr un SQL que escribió un
   modelo?
2. ¿Por qué se usa temperatura 0 para generar SQL, y por qué eso no garantiza
   que la respuesta sea correcta?
3. Si el modelo solo recibe los nombres de las columnas, ¿qué tipo de errores
   esperas en el SQL que genera, y cómo los detectarías antes de que lleguen a
   negocio?